In [ ]:
# UltraIS Kaggle setup: dependencies
!pip install -q einops timm==0.9.16 transformers==4.38.1 safetensors \
    tensorboard opencv-python pyyaml matplotlib==3.7.5 \
    scipy==1.11.4 tqdm huggingface-hub==0.21.1 regex addict lmdb \
    gdown rawpy ipdb yacs scikit-image lpips

Processing /kaggle/input/datasets/vphanch/new-wheel/causal_conv1d-1.5.3-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/vphanch/new-wheel/mamba_ssm-2.2.6-cp312-cp312-linux_x86_64.whl


In [ ]:
# Clone repos
!git clone https://github.com/Monarl/UltraIS-main.git
!git clone https://github.com/Monarl/LLLR-Image-Enhancement.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 39.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 84.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 85.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 52.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.1/346.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.9 MB/s

In [ ]:
# Download and extract RELLISUR into the same location UltraIS config expects
!python -m gdown --id 1ObTHkvT4ld4fah2bZ22343gF3d-NRUid -O /tmp/rellisur.zip
!unzip -qo /tmp/rellisur.zip -d /kaggle/working/LLLR-Image-Enhancement/datasets/
!rm /tmp/rellisur.zip

Cloning into 'LLLR-Image-Enhancement'...
remote: Enumerating objects: 445, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 445 (delta 55), reused 47 (delta 39), pack-reused 372 (from 2)
Receiving objects: 100% (445/445), 191.56 MiB | 45.43 MiB/s, done.
Resolving deltas: 100% (203/203), done.
Updating files: 100% (241/241), done.


In [ ]:
import glob
import shutil
from pathlib import Path
import yaml

ultra_root = Path("/kaggle/working/UltraIS-main")
data_root = Path("/kaggle/working/LLLR-Image-Enhancement/datasets/RELLISUR-Dataset")
pretrained_dir = ultra_root / "experiments/pretrained_models"
pretrained_dir.mkdir(parents=True, exist_ok=True)

# ---------- Find UltraIS x2 checkpoint ----------
ckpt_candidates = []
ckpt_candidates += glob.glob(str(pretrained_dir / "net_g_*x2*.pth"))
ckpt_candidates += glob.glob("/kaggle/input/**/*x2*.pth", recursive=True)
ckpt_candidates = sorted(set(ckpt_candidates))

if not ckpt_candidates:
    raise FileNotFoundError(
        "No x2 checkpoint found. Upload your .pth to /kaggle/input or put it in UltraIS-main/experiments/pretrained_models"
    )

ckpt = Path(ckpt_candidates[-1])
print("Using checkpoint:", ckpt)

# ---------- Find and place HRNet pretrained ----------
# UltraIS expects this exact filename under experiments/pretrained_models/
hrnet_target = pretrained_dir / "hrnet_w48_pascal_context_cls59_480x480.pth"
hrnet_candidates = []
hrnet_candidates += glob.glob(str(hrnet_target))
hrnet_candidates += glob.glob("/kaggle/input/**/*hrnet*w48*pascal*480x480*.pth", recursive=True)
hrnet_candidates += glob.glob("/kaggle/input/**/*hrnet*.pth", recursive=True)
hrnet_candidates = sorted(set(hrnet_candidates))

if not hrnet_candidates:
    raise FileNotFoundError(
        "HRNet pretrained not found. Please upload hrnet_w48_pascal_context_cls59_480x480.pth to /kaggle/input."
    )

hrnet_src = Path(hrnet_candidates[-1])
if hrnet_src.resolve() != hrnet_target.resolve():
    shutil.copy2(hrnet_src, hrnet_target)
print("Using HRNet pretrained:", hrnet_target)

cfg = {
    "name": "test_ultrais_x2_kaggle",
    "model_type": "CSDLLSRv4",
    "scale": 2,
    "num_gpu": 1,
    "manual_seed": 100,
    "datasets": {
        "test_1": {
            "name": "RELLISUR_Test_X2",
            "type": "Dataset_PairedImage",
            "dataroot_gt": str(data_root / "Test/NLHR/X2"),
            "dataroot_lq": str(data_root / "Test/LLLR"),
            "direct_gt_match": True,
            "io_backend": {"type": "disk"},
            "use_grayatten": False,
            "use_illguidance": True
        }
    },
    "network_g": {
        "type": "CSDLLSRNetv9_7_5",
        "inp_channels": 3,
        "out_channels": 3,
        "n_feat": 64,
        "scale": 2
    },
    "path": {
        "pretrain_network_g": str(ckpt),
        "strict_load_g": True,
        "param_key": "params"
    },
    "val": {
        "window_size": 16,
        "save_img": True,
        "rgb2bgr": True,
        "use_image": True,
        "metrics": {
            "psnr": {"type": "calculate_psnr", "crop_border": 2, "test_y_channel": False},
            "ssim": {"type": "calculate_ssim", "crop_border": 2, "test_y_channel": False}
        }
    },
    "train": {"total_iter": 1},
    "logger": {
        "print_freq": 100,
        "save_checkpoint_freq": 100,
        "use_tb_logger": False,
        "wandb": {"project": None, "resume_id": None}
    },
    "dist_params": {"backend": "nccl", "port": 29500}
}

opt_path = ultra_root / "Super_Resolution/Options/CSDLLSR_v9_7_5_3_scale2_test_kaggle.yml"
opt_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Wrote:", opt_path)

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1ObTHkvT4ld4fah2bZ22343gF3d-NRUid
From (redirected): https://drive.google.com/uc?id=1ObTHkvT4ld4fah2bZ22343gF3d-NRUid&confirm=t&uuid=2cb5d904-d6f2-4c93-8de5-f7e33d26e3a2
To: /tmp/rellisur.zip
100%|██████████████████████████████████████| 8.86G/8.86G [01:44<00:00, 84.7MB/s]


In [ ]:
# Run UltraIS x2 test inference
%cd /kaggle/working/UltraIS-main
!PYTHONPATH=/kaggle/working/UltraIS-main CUDA_VISIBLE_DEVICES=0 python -m basicsr.test \
    -opt Super_Resolution/Options/CSDLLSR_v9_7_5_3_scale2_test_kaggle.yml

Wrote (overwrote): /kaggle/working/LLLR-Image-Enhancement/options/test/mambairv2_llie_sr/test_MambaIRv2_LLIESR_x2.yml
['name: test_MambaIRv2_LLIESR_x2', 'model_type: MambaIRv2LLIESRModel', 'scale: 2', 'num_gpu: 1', 'manual_seed: 10', 'datasets:', '  test_1:', '    name: RELLISUR_Test', '    type: PairedImageDataset', '    dataroot_gt: datasets/RELLISUR-Dataset/Test/NLHR/X2', '    dataroot_lq: datasets/RELLISUR-Dataset/Test/LLLR', '    direct_gt_match: true', '    use_illguidance: true', '    io_backend:', '      type: disk', 'network_g:', '  type: MambaIRv2LLIESR', '  upscale: 2', '  in_chans: 3', '  img_size: 64', '  img_range: 1.0', '  embed_dim: 48', '  d_state: 8', '  depths:', '  - 5', '  - 5', '  - 5', '  - 5', '  num_heads:', '  - 4', '  - 4', '  - 4', '  - 4', '  window_size: 16', '  inner_rank: 32', '  num_tokens: 64', '  convffn_kernel_size: 5', '  mlp_ratio: 1.0', '  upsampler: pixelshuffledirect', '  resi_connection: 1conv', '  igm_n_feat: 48', '  isdm_num_heads: 2', '  isd

In [ ]:
# Optional: compute full metrics CSV (PSNR/SSIM/LPIPS/RMSE/NIQE/LOE)
%cd /kaggle/working/UltraIS-main
!PYTHONPATH=/kaggle/working/UltraIS-main python metric_full_eval_out2csv.py \
    --pred_dir results/test_ultrais_x2_kaggle/visualization/RELLISUR_Test_X2 \
    --gt_dir /kaggle/working/LLLR-Image-Enhancement/datasets/RELLISUR-Dataset/Test/NLHR/X2 \
    --lq_dir /kaggle/working/LLLR-Image-Enhancement/datasets/RELLISUR-Dataset/Test/LLLR \
    --csv_path results/test_ultrais_x2_kaggle/full_metrics.csv \
    --crop_border 2

/kaggle/working/LLLR-Image-Enhancement
Disable distributed.
2026-04-10 03:12:38,817 INFO: 
  name: test_MambaIRv2_LLIESR_x2
  model_type: MambaIRv2LLIESRModel
  scale: 2
  num_gpu: 1
  manual_seed: 10
  datasets:[
    test_1:[
      name: RELLISUR_Test
      type: PairedImageDataset
      dataroot_gt: datasets/RELLISUR-Dataset/Test/NLHR/X2
      dataroot_lq: datasets/RELLISUR-Dataset/Test/LLLR
      direct_gt_match: True
      use_illguidance: True
      io_backend:[
        type: disk
      ]
      phase: test
      scale: 2
    ]
  ]
  network_g:[
    type: MambaIRv2LLIESR
    upscale: 2
    in_chans: 3
    img_size: 64
    img_range: 1.0
    embed_dim: 48
    d_state: 8
    depths: [5, 5, 5, 5]
    num_heads: [4, 4, 4, 4]
    window_size: 16
    inner_rank: 32
    num_tokens: 64
    convffn_kernel_size: 5
    mlp_ratio: 1.0
    upsampler: pixelshuffledirect
    resi_connection: 1conv
    igm_n_feat: 48
    isdm_num_heads: 2
    isdm_ffn_expansion: 2.66
  ]
  path:[
    pretrain_netw